# Select dataset

To make our lives a little bit easier, we only choose images that have at least one human face in them. <br>
To do so, we use a pre-trained facial detection model: **Insightface SCRFD.** <br>
This model will automatically classify all images with faces.

## Prepare original dataset

In [1]:
cd "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel/"

/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel


In [2]:
import pandas as pd

In [3]:
#read in dataset
df_dataset = pd.read_csv("posts_act.csv",
                        
                         sep = ";") #specify separator, otherwise the parser will cry

#verify
df_dataset.head()

,link.x,text,created_time,post_type,language_text.iso_lang_1,platform_name,likes,comments,user_name,handle,filename,media_text
0,https://www.instagram.com/p/C0zSLkGpiNx/,Some of my favorite frames from my short movie...,2023-12-13 17:18:06,album,en,instagram,215.0,17.0,•ᴛᴀᴛɪᴀɴᴀ•,tanicka_000,/Reichel/media/1476906_781821_image.jpeg,NaN
1,https://www.instagram.com/p/C0zORzNInqe/,Parallel zur #COP28 haben wir uns die letzten ...,2023-12-13 16:44:00,album,de,instagram,70.0,1.0,BUNDjugend,bundjugend,/Reichel/media/1472492_778909_image.jpeg,NaN
2,https://www.instagram.com/p/C0zSt80tzXo/,🤝🚍 Wir Fahren Zusammen ist eine Kampagne von @...,2023-12-13 17:22:48,photo,de,instagram,105.0,3.0,FridaysForFuture - BieleFFFeld,fridaysforfuture.bielefeld,/Reichel/media/1472605_778974_image.jpeg,NaN
3,https://www.instagram.com/p/C0zaosQIYVd/,🥳 KLIMANEUTRAL 2040?! Yeah! 🤩🤗😍\n\n++ Niedersa...,2023-12-13 18:31:59,photo,de,instagram,153.0,9.0,FridaysForFuture Hannover 🌍,fridaysforfuture_hannover,/Reichel/media/1472738_779046_image.jpeg,FUTURE FUTURE WER STRABEK SAT WIRD PROTEST ERN...
4,https://www.instagram.com/p/C0zZIEFIIFE/,"As COP28 concludes, Kitty van der Heijden, UNI...",2023-12-13 18:18:48,photo,en,instagram,1657.0,17.0,UNICEF,unicef,/Reichel/media/1474505_780204_image.jpeg,"Another future is still possible. A fast, fair..."


## Fix filename for correct absolute path

We can use `Series.replace()` or `str.replace()`. <br>
We are going to use `str.replace()` as it is more efficient and faster (note: does not really matter for a small dataset like this).

In [4]:
#fix filename
df_dataset["filename"] = df_dataset["filename"].str.replace(
    "/Reichel/media/",
    "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel/media_act/",
    regex = False
)

In [5]:
#export column to list

image_paths = df_dataset["filename"].tolist()

print(type(image_paths))
print(len(image_paths))

<class 'list'>
8190


## Iterate through images

In [6]:
import os
import shutil
import cv2
import onnxruntime
from insightface.app import FaceAnalysis as FA
from PIL import Image, ImageOps

In [7]:
#create new directory
output_dir = "images_with_faces"
os.makedirs(output_dir, exist_ok = True)

#create app
app = FA(

    #use CUDA device = NVIDIA Blackwell GB203
    providers=["CUDAExecutionProvider"]
)

app.prepare(ctx_id = 0)

#define function for iteration
def has_face(img_path):
    try:
        #read image from path
        img = cv2.imread(img_path)
        faces = app.get(img)

        if len(faces) > 0: #means at least one face exists

            #copy to directory
            shutil.copy2(
                img_path,
                os.path.join(output_dir, os.path.basename(img_path))
            )
            return True

        #if not len > 0 -> means no face exists
        return False

    #if exception
    except Exception:
        return False

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/simon/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], 

In [9]:
#iterate over all images
for path in image_paths:
    has_face(path)

/home/simon/anaconda3/envs/findingemo_download/lib/python3.13/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
